In [1]:
pip install duckdb streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 45.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 52.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 31.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 51.2 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.9 MB/s  0:00:006m0:00:01
  Attempting uninstall: narwhalsm━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [protobuf]
    Found existing installation: narwhals 1.38.0━━━━━━━━━━━━━━  4/14 [protobuf]
    Uninstalling narwhals-1.38.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [protobuf]
      Successfully uninstalled narwhals-1.38.0━━━━━━━━━━━━━━━━  4/14 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [streamlit]14 [streamlit]
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import duckdb
import streamlit as st

DB_PATH = "app.duckdb"

@st.cache_resource                      # one connection per server process
def get_con():
    con = duckdb.connect(DB_PATH)
    con.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY,
            author VARCHAR,
            body VARCHAR,
            created TIMESTAMP DEFAULT now()
        )
    """)
    con.execute("CREATE SEQUENCE IF NOT EXISTS notes_seq START 1")
    return con

con = get_con()

# --- write ---
with st.form("add"):
    author = st.text_input("Author")
    body = st.text_area("Note")
    if st.form_submit_button("Save") and body:
        con.execute(
            "INSERT INTO notes (id, author, body) VALUES (nextval('notes_seq'), ?, ?)",
            [author, body],
        )
        st.cache_data.clear()           # drop cached reads so the new row shows

# --- read ---
@st.cache_data(ttl=60)
def load_notes():
    return con.execute("SELECT * FROM notes ORDER BY created DESC").df()

st.dataframe(load_notes())

d

2026-11-15 13:29:36.148 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:29:36.185 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:29:36.186 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:29:36.187 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:29:36.187 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:29:36.188 WARNING streamlit.run

DeltaGenerator(_form_data=FormData(form_id='add'))

In [ ]:
display(st.dataframe)

2026-11-15 13:38:20.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:38:20.276 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:38:20.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:38:20.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-15 13:38:20.279 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator(_form_data=FormData(form_id='add'))

In [1]:
pip install libsql

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
environ = os.environ["TURSO_URL"]
print(environ)

libsql://myturso-nkletnoi.aws-us-east-1.turso.io


In [1]:
import os, libsql
conn = libsql.connect("myturso.db", sync_url=os.environ["TURSO_URL"], auth_token=os.environ["TURSO_TOKEN"])
conn.sync()

In [8]:
# conn.execute("CREATE TABLE IF NOT EXISTS notes (id INTEGER PRIMARY KEY, body TEXT)")
# conn.execute("INSERT INTO notes (body) VALUES (?)", ("hello from codespaces",))
# conn.commit()
# conn.sync()   # pushes the write to Turso

import pandas as pd
df = pd.DataFrame(conn.execute("SELECT date as date, region as region, revenue as revenue FROM sales_explorer_source").fetchall(), columns=["date", "region", "revenue"])
df

,date,region,revenue
0,2026-01-01,East,1045.71
1,2026-01-02,East,844.00
2,2026-01-03,East,1112.57
3,2026-01-04,East,1141.08
4,2026-01-05,East,707.34
...,...,...,...
535,2026-06-25,South,895.62
536,2026-06-26,South,719.18
537,2026-06-27,South,972.17
538,2026-06-28,South,440.86


In [6]:
import os, re, libsql
import pandas as pd

# ---- settings ----
CSV_PATH = "sales_explorer_source.csv"      # file you uploaded to the codespace
REPLACE_IF_EXISTS = True                    # True = drop & reload, False = append


# ---- read file & derive names ----
df = pd.read_csv(CSV_PATH)

def clean(name):
    return re.sub(r"\W+", "_", str(name).strip()).strip("_").lower()

table = clean(os.path.splitext(os.path.basename(CSV_PATH))[0])   # -> sales_explorer_source
df.columns = [clean(c) for c in df.columns]                      # -> date, region, revenue

def sqlite_type(dtype):
    if pd.api.types.is_integer_dtype(dtype): return "INTEGER"
    if pd.api.types.is_float_dtype(dtype):   return "REAL"
    return "TEXT"          # dates stay as ISO text (YYYY-MM-DD), which SQLite sorts correctly

# ---- create table ----
if REPLACE_IF_EXISTS:
    conn.execute(f'DROP TABLE IF EXISTS "{table}"')

col_defs = ", ".join(f'"{c}" {sqlite_type(t)}' for c, t in df.dtypes.items())
conn.execute(f'CREATE TABLE IF NOT EXISTS "{table}" ({col_defs})')

# ---- insert rows ----
col_names = ", ".join(f'"{c}"' for c in df.columns)
placeholders = ", ".join("?" for _ in df.columns)
rows = [tuple(None if pd.isna(v) else v for v in r)
        for r in df.itertuples(index=False, name=None)]

conn.executemany(f'INSERT INTO "{table}" ({col_names}) VALUES ({placeholders})', rows)
conn.commit()
conn.sync()                                 # push to Turso

print(f"Table '{table}' created with columns: {list(df.columns)}")
print(f"Rows loaded: {len(rows)}")

Table 'sales_explorer_source' created with columns: ['date', 'region', 'revenue']
Rows loaded: 540


In [ ]:
import duckdb
con.close()

c = duckdb.connect("lake/warehouse.duckdb", read_only=True)
c.sql("SELECT * FROM ctl.watermark ORDER BY name").show()
c.sql("SELECT * FROM ctl.batch_log").show()
c.close()

IOException: IO Error: Could not set lock on file "/workspaces/pythonProject/lake/warehouse.duckdb": Conflicting lock is held in /usr/local/python/3.12.1/bin/python3.12 (PID 14764). See also https://duckdb.org/docs/current/connect/concurrency

In [1]:
import pyspark; print(pyspark.__version__)

4.3.0


In [2]:
pd.DataFrame(conn.execute(f'SELECT region, COUNT(*), ROUND(SUM(revenue),2) FROM sales_explorer_source GROUP BY region').fetchall(),
             columns=["region", "rows", "total_revenue"])

NameError: name 'pd' is not defined